# ATSEG Segmentation Imputation — Multi-class SVM Pipeline
## VELSIPITY Pharma Commercial Analytics

**Objective:** Reconstruct missing ATSEG segmentation labels for doctors using a Multi-class Support Vector Machine (SVM) classifier, trained on the user-level collapsed dataset.

**Transparency note:** ATSEG was originally constructed from behavioral prescribing and promotional signals. This model *reconstructs an existing segmentation rule* — not discovering new latent structure. All predictions represent inferred segment membership based on behavioral similarity to labeled doctors.

---
### Notebook Sections
1. Project Overview
2. Data Loading
3. Data Understanding
4. Target Definition
5. Feature Selection & Leakage Review
6. Class Balancing Strategy
7. Preprocessing Pipeline
8. Linear SVM Training
9. RBF SVM Training
10. Comparative Evaluation
11. Model Interpretation
12. Population Shift Diagnosis
13. Prediction on Missing Users
14. Final Dataset Export
15. Conclusions & Limitations


## 1. Project Overview

**Business Goal:** Complete missing ATSEG assignments for 9,032 unlabeled doctors so every doctor in the commercial database has a segment label.

**Model Choice:** Multi-class SVM evaluating two kernels:
- Linear kernel: fast, interpretable, good for high-dimensional data
- RBF kernel: captures non-linearities, provides calibrated probabilities

**Key design decisions:**
- Training only on 11,899 labeled doctors
- Random undersampling to address class imbalance
- StandardScaler mandatory for SVM correctness
- Platt scaling for probability outputs on RBF model
- No re-aggregation: dataset is already at user level (one row per doctor)


In [62]:
# Setup & Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

INPUT_PATH = 'doctors_means.csv'
OUTPUT_CSV = 'doctors_svm_atseg_completed.csv'

print('All imports OK')
import sklearn; print(f'  sklearn {sklearn.__version__}')
print(f'  pandas  {pd.__version__}')


All imports OK
  sklearn 1.5.0
  pandas  2.2.2


## 2. Data Loading

Load the user-level collapsed dataset. Each row is one doctor (`NUEVO_ID`).

In [63]:
df = pd.read_csv(INPUT_PATH)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
df.head(3)


Shape: 20,931 rows x 70 columns
Memory: 15.3 MB


,NUEVO_ID,SPEC_GE_first,SPEC_GPFM_first,SPEC_IM_first,SPEC_NRP_first,SPEC_OTHER_SPEC_first,SPEC_PHA_first,STATE_1_first,STATE_2_first,STATE_3_first,...,DIRECTMAIL,SPK,DETAILS,ENGAGEMENT_SCORE,BRAND1_NTB_GIDX,BRAND2_NTB_GIDX,BRAND1_T_GIDX,BRAND2_T_GIDX,NBRx_RATIO,BRAND1_MARKET_SHARE
0,1,-1.0,0.0,0.0,0.0,0.0,0.576582,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.069767
1,2,-1.0,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.313953
2,3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.015259,0.0,0.146799,0.162058,0.0,0.0,0.0,0.0,0.0,0.709302


## 3. Data Understanding

Comprehensive diagnostics: duplicates, missingness, target distribution, and feature inventory.

In [64]:
# 3.1 Duplicate NUEVO_ID check
dupes = df['NUEVO_ID'].duplicated().sum()
print(f'Duplicate NUEVO_ID: {dupes}')
assert dupes == 0, 'ERROR: duplicates found!'
print('  OK - one row per doctor confirmed')


Duplicate NUEVO_ID: 0
  OK - one row per doctor confirmed


In [65]:
# 3.2 Missingness audit
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
print(f'Columns with missing values: {len(missing_cols)}')
if len(missing_cols) == 0:
    print('  OK - No missing values in raw features')
else:
    print(missing_cols)


Columns with missing values: 0
  OK - No missing values in raw features


In [66]:
# 3.3 ATSEG_first raw distribution
print('ATSEG_first raw distribution:')
dist = df['ATSEG_first'].value_counts(dropna=False)
print(dist)
print()
print("Note: '0' encodes missing ATSEG (no actual NaN in raw file)")
labeled_n   = dist[dist.index != '0'].sum()
unlabeled_n = dist.get('0', 0)
total = len(df)
print(f'  Labeled   : {labeled_n:,}  ({labeled_n/total:.1%})')
print(f'  Unlabeled : {unlabeled_n:,}  ({unlabeled_n/total:.1%})')


ATSEG_first raw distribution:
ATSEG_first
0        9032
SEG_A    6406
SEG_B    3349
SEG_C    2144
Name: count, dtype: int64

Note: '0' encodes missing ATSEG (no actual NaN in raw file)
  Labeled   : 11,899  (56.8%)
  Unlabeled : 9,032  (43.2%)


In [67]:
# 3.4 Feature inventory
EXCLUDE = ['NUEVO_ID','ATSEG_first','ATSEG','ATSEG_IS_MISSING',
           'WEEK_ID_first','WEEK_ID_last','WEEK_ID_count']
feat_cols = [c for c in df.columns if c not in EXCLUDE and df[c].dtype != object]
print(f'Feature columns: {len(feat_cols)}')
groups = {
    'TRX/NRX volume':     [c for c in feat_cols if 'TRX' in c or 'NRX' in c],
    'Claims':             [c for c in feat_cols if 'CLM' in c],
    'Promotion':          [c for c in feat_cols if any(x in c for x in ['RTE','SAMPLE','COPAY','DIRECTMAIL','SPK','DETAIL','ENGAGEMENT'])],
    'Growth indices':     [c for c in feat_cols if 'GIDX' in c or 'MARKET_SHARE' in c or 'NBRx_RATIO' in c],
    'Specialty mix':      [c for c in feat_cols if 'SPEC_' in c],
    'Geography mix':      [c for c in feat_cols if 'STATE_' in c or 'STS_' in c],
    'Cohort bins':        [c for c in feat_cols if '1940' in c or '1960' in c or '1980' in c],
}
print()
for grp, cols in groups.items():
    print(f'  {grp:<25}: {len(cols):>3} columns')


Feature columns: 65

  TRX/NRX volume           :  13 columns
  Claims                   :  15 columns
  Promotion                :   7 columns
  Growth indices           :   6 columns
  Specialty mix            :   6 columns
  Geography mix            :   9 columns
  Cohort bins              :   3 columns


In [68]:
# 3.5 WEEK_ID_count - confirm constant
print('WEEK_ID_count unique values:', df['WEEK_ID_count'].unique())
print('  -> All doctors have 86 weeks (balanced panel)')


WEEK_ID_count unique values: [86]
  -> All doctors have 86 weeks (balanced panel)


## 4. Target Definition

- `ATSEG_first = '0'` encodes missing ATSEG. We convert to `NaN` and create a binary flag.
- **Labeled subset:** `ATSEG_IS_MISSING == 0` — used for training and validation
- **Unlabeled subset:** `ATSEG_IS_MISSING == 1` — will receive predicted labels


In [69]:
# Standardize target encoding
df['ATSEG']            = df['ATSEG_first'].replace({'0': np.nan})
df['ATSEG_IS_MISSING'] = (df['ATSEG_first'] == '0').astype(int)

labeled   = df[df['ATSEG_IS_MISSING'] == 0].copy()
unlabeled = df[df['ATSEG_IS_MISSING'] == 1].copy()

print(f'Labeled doctors   : {len(labeled):,}')
print(f'Unlabeled doctors : {len(unlabeled):,}')

# Validate flag vs null consistency
mismatch = (
    ((df['ATSEG_IS_MISSING']==0) & df['ATSEG'].isnull()) |
    ((df['ATSEG_IS_MISSING']==1) & df['ATSEG'].notna())
).sum()
print(f'Flag vs null mismatches: {mismatch}  OK' if mismatch==0 else f'MISMATCH: {mismatch}')


Labeled doctors   : 11,899
Unlabeled doctors : 9,032
Flag vs null mismatches: 0  OK


In [70]:
# Class distribution (labeled only)
print('Class distribution in labeled subset:')
class_dist = labeled['ATSEG'].value_counts()
for cls, cnt in class_dist.items():
    bar = '#' * int(cnt/100)
    print(f'  {cls}: {cnt:>5,} ({cnt/len(labeled):.1%})  {bar}')

ratio = class_dist['SEG_A'] / class_dist['SEG_C']
print(f'\nImbalance ratio SEG_A/SEG_C: {ratio:.2f}x')
print('-> Moderate imbalance (~3:1). SVM requires explicit balancing strategy.')


Class distribution in labeled subset:
  SEG_A: 6,406 (53.8%)  ################################################################
  SEG_B: 3,349 (28.1%)  #################################
  SEG_C: 2,144 (18.0%)  #####################

Imbalance ratio SEG_A/SEG_C: 2.99x
-> Moderate imbalance (~3:1). SVM requires explicit balancing strategy.


## 5. Feature Selection & Leakage Review

### Columns excluded and why

| Column | Reason |
|---|---|
| `NUEVO_ID` | Identifier - not a predictor |
| `ATSEG_first` | Raw target column |
| `ATSEG` | Derived target |
| `ATSEG_IS_MISSING` | Target flag |
| `WEEK_ID_first`, `WEEK_ID_last` | Date strings, no predictive value |
| `WEEK_ID_count` | Constant = 86 for all rows (zero variance) |

### Leakage assessment

**Unacceptable leakage:** None identified. No column directly encodes ATSEG.

**Acceptable reconstruction:** All features are behavioral aggregates (TRX, NRX, promotions, specialty, geography) - the same signals that originally created ATSEG. Using them is appropriate and is explicitly acknowledged as reconstruction.

**Multicollinearity:** 215+ pairs with |r| > 0.90 (e.g., UC_TRX_sum, _mean, _max). Handled by SVM regularization (C parameter), not manual removal.


In [71]:
# Confirm feature set
print(f'Final feature set: {len(feat_cols)} numeric columns')
print('All numeric:', all(df[c].dtype in [np.float64, np.int64] for c in feat_cols))

# Correlation overview
corr = labeled[feat_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
n_high = int((upper > 0.90).sum().sum())
print(f'Feature pairs |r|>0.90: {n_high}')
print('  -> Will be managed by L2 regularization inside SVM (C parameter)')


Final feature set: 65 numeric columns
All numeric: True
Feature pairs |r|>0.90: 38
  -> Will be managed by L2 regularization inside SVM (C parameter)


## 6. Class Balancing Strategy

### Why balancing is critical for SVM

SVM maximizes the margin between classes. With imbalanced classes, the decision boundary is biased toward the majority class (SEG_A), causing underprediction of minority classes. Two approaches are possible:

1. **`class_weight='balanced'`** in SVM — adjusts loss weights inversely proportional to class frequency
2. **Random undersampling** — downsample majority classes to match the minority class count

We use **both** (undersampling + `class_weight='balanced'`) for maximum balance. This combination:
- Reduces computational cost (SVM is O(n^2)-O(n^3))
- Produces a balanced support vector set
- Avoids synthetic sample risk (SMOTE in high-dimensional space may create unrealistic interpolations)

**Critical rule:** Balancing is applied ONLY on the training split, never on the test set.

**Trade-off:** Random undersampling discards labeled majority-class data. The test set reflects the original (unbalanced) distribution for realistic evaluation.


In [72]:
# 6.1 Train/Test split (stratified)
X_labeled = labeled[feat_cols]
y_labeled  = labeled['ATSEG']

X_train, X_test, y_train, y_test = train_test_split(
    X_labeled, y_labeled,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_labeled,
)

print('Train/Test split (stratified by ATSEG):')
print(f'  Train : {len(X_train):,}')
print(f'  Test  : {len(X_test):,}')
print()
print('Class balance:')
for name, y_s in [('Train', y_train), ('Test', y_test)]:
    print(f'  {name}:')
    for cls in y_labeled.unique():
        n = (y_s==cls).sum()
        print(f'    {cls}: {n:,} ({n/len(y_s):.1%})')


Train/Test split (stratified by ATSEG):
  Train : 9,519
  Test  : 2,380

Class balance:
  Train:
    SEG_A: 5,125 (53.8%)
    SEG_B: 2,679 (28.1%)
    SEG_C: 1,715 (18.0%)
  Test:
    SEG_A: 1,281 (53.8%)
    SEG_B: 670 (28.2%)
    SEG_C: 429 (18.0%)


In [73]:
# 6.2 Random undersampling on TRAIN ONLY
min_per_class = y_train.value_counts().min()
print(f'Minority class size: {min_per_class:,} (SEG_C)')
print(f'Strategy: downsample SEG_A and SEG_B to {min_per_class:,} samples each')

balanced_idx = pd.concat([
    y_train[y_train == cls].sample(min_per_class, random_state=RANDOM_SEED)
    for cls in y_train.unique()
]).index

X_train_bal = X_train.loc[balanced_idx]
y_train_bal = y_train.loc[balanced_idx]

print()
print('Original train distribution:')
for cls, n in y_train.value_counts().items():
    print(f'  {cls}: {n:,}')
print()
print('Balanced train distribution:')
for cls, n in y_train_bal.value_counts().items():
    print(f'  {cls}: {n:,}')
print(f'Total balanced: {len(X_train_bal):,}')
print(f'Samples discarded: {len(X_train)-len(X_train_bal):,}')


Minority class size: 1,715 (SEG_C)
Strategy: downsample SEG_A and SEG_B to 1,715 samples each

Original train distribution:
  SEG_A: 5,125
  SEG_B: 2,679
  SEG_C: 1,715

Balanced train distribution:
  SEG_A: 1,715
  SEG_C: 1,715
  SEG_B: 1,715
Total balanced: 5,145
Samples discarded: 4,374


## 7. Preprocessing Pipeline

### Why StandardScaler is mandatory for SVM

SVM finds the maximum-margin hyperplane using Euclidean distances. Without scaling, features with large ranges (e.g., UC_TRX_sum: 0-650) dominate those with small ranges (e.g., SPEC_GE_first: 0/-1), regardless of predictive relevance. StandardScaler ensures all features contribute equally.

**Pipeline:**
1. Median imputation (safety layer - no actual missing values)
2. StandardScaler (zero-mean, unit-variance)

The preprocessor is fitted on the BALANCED TRAINING SET only, then applied to test and unlabeled sets.


In [74]:
# Build and fit preprocessing pipeline
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Fit on balanced training ONLY
X_train_s = preprocessor.fit_transform(X_train_bal)
X_test_s  = preprocessor.transform(X_test)
X_unlab_s = preprocessor.transform(unlabeled[feat_cols])

print('Preprocessing applied:')
print(f'  Balanced train : {X_train_s.shape}')
print(f'  Test           : {X_test_s.shape}')
print(f'  Unlabeled      : {X_unlab_s.shape}')
print(f'Post-scale mean  : {X_train_s[:,:3].mean(axis=0).round(4)}')
print(f'Post-scale std   : {X_train_s[:,:3].std(axis=0).round(4)}')
print('OK - Features correctly standardized (mean~0, std~1)')


Preprocessing applied:
  Balanced train : (5145, 65)
  Test           : (2380, 65)
  Unlabeled      : (9032, 65)
Post-scale mean  : [ 0.  0. -0.]
Post-scale std   : [1. 1. 1.]
OK - Features correctly standardized (mean~0, std~1)


## 8. Linear SVM Training

**LinearSVC** uses One-vs-Rest (OvR) multi-class strategy by default:
- Trains one binary classifier per class (A-vs-rest, B-vs-rest, C-vs-rest)
- Final prediction = class with highest decision score
- Fast and interpretable via coefficients
- Tuning: C across [0.01, 0.1, 1.0, 10.0]


In [75]:
print('Training LinearSVC across C values...')
print(f'{"C":>8} | {"Macro-F1":>9} | {"Weighted-F1":>11} | {"Bal-Acc":>8}')
print('-' * 46)

lin_results = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = LinearSVC(C=C, max_iter=3000, random_state=RANDOM_SEED, class_weight='balanced')
    clf.fit(X_train_s, y_train_bal)
    yp = clf.predict(X_test_s)
    mf1 = f1_score(y_test, yp, average='macro')
    wf1 = f1_score(y_test, yp, average='weighted')
    ba  = balanced_accuracy_score(y_test, yp)
    lin_results[C] = {'mf1': mf1, 'wf1': wf1, 'ba': ba, 'clf': clf}
    print(f'{C:>8} | {mf1:>9.4f} | {wf1:>11.4f} | {ba:>8.4f}')

best_lin_C = max(lin_results, key=lambda c: lin_results[c]['mf1'])
best_lin   = lin_results[best_lin_C]
print(f'\nBest Linear: C={best_lin_C}  Macro-F1={best_lin["mf1"]:.4f}')


Training LinearSVC across C values...
       C |  Macro-F1 | Weighted-F1 |  Bal-Acc
----------------------------------------------
    0.01 |    0.5598 |      0.6280 |   0.5651
     0.1 |    0.5622 |      0.6300 |   0.5679
     1.0 |    0.5632 |      0.6310 |   0.5693
    10.0 |    0.5619 |      0.6299 |   0.5680

Best Linear: C=1.0  Macro-F1=0.5632


In [76]:
# Detailed Linear SVM evaluation
clf_lin = best_lin['clf']
yp_lin  = clf_lin.predict(X_test_s)

print('=== Linear SVM - Hold-out Evaluation ===')
print(f'C={best_lin_C} | Strategy: One-vs-Rest | max_iter=3000')
print()
print(f'Macro F1    : {best_lin["mf1"]:.4f}')
print(f'Weighted F1 : {best_lin["wf1"]:.4f}')
print(f'Balanced Acc: {best_lin["ba"]:.4f}')
print()
print(classification_report(y_test, yp_lin))
cm_lin = confusion_matrix(y_test, yp_lin, labels=['SEG_A','SEG_B','SEG_C'])
print('Confusion Matrix:')
print(pd.DataFrame(cm_lin, index=['SEG_A','SEG_B','SEG_C'], columns=['SEG_A','SEG_B','SEG_C']))


=== Linear SVM - Hold-out Evaluation ===
C=1.0 | Strategy: One-vs-Rest | max_iter=3000

Macro F1    : 0.5632
Weighted F1 : 0.6310
Balanced Acc: 0.5693

              precision    recall  f1-score   support

       SEG_A       0.77      0.78      0.78      1281
       SEG_B       0.54      0.43      0.48       670
       SEG_C       0.38      0.50      0.43       429

    accuracy                           0.63      2380
   macro avg       0.57      0.57      0.56      2380
weighted avg       0.64      0.63      0.63      2380

Confusion Matrix:
       SEG_A  SEG_B  SEG_C
SEG_A   1000    134    147
SEG_B    188    287    195
SEG_C    107    108    214


In [77]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(y_true, y_pred, classes,
                         normalize=False,
                         title='Confusion Matrix'):
    
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
        title = title + " (Normalized)"
    
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], fmt),
                     ha="center",
                     va="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

## 9. RBF SVM Training

**SVC with RBF kernel** uses One-vs-One (OvO) multi-class strategy (sklearn default):
- Trains one binary classifier per class pair: A-vs-B, A-vs-C, B-vs-C (3 classifiers total)
- Final prediction by majority voting
- Advantage: each sub-classifier sees balanced binary problems
- `probability=True`: enables Platt scaling for calibrated probability outputs (required for confidence scores)
- `gamma='scale'` = 1/(n_features * X.var()): automatically adapts to feature variance
- Note: Platt scaling adds computational overhead (5-fold internal CV per sub-classifier)


In [78]:
print('Training RBF SVC across (C, gamma) combinations...')
print(f'{"C":>6} {"gamma":>8} | {"Macro-F1":>9} | {"Weighted-F1":>11} | {"Bal-Acc":>8}')
print('-' * 52)

rbf_results = {}
for C, gamma in [(0.1, 'scale'), (1.0, 'scale'), (10.0, 'scale'), (1.0, 'auto')]:
    clf = SVC(C=C, gamma=gamma, kernel='rbf', probability=True,
              random_state=RANDOM_SEED, class_weight='balanced')
    clf.fit(X_train_s, y_train_bal)
    yp = clf.predict(X_test_s)
    mf1 = f1_score(y_test, yp, average='macro')
    wf1 = f1_score(y_test, yp, average='weighted')
    ba  = balanced_accuracy_score(y_test, yp)
    rbf_results[(C, gamma)] = {'mf1': mf1, 'wf1': wf1, 'ba': ba, 'clf': clf}
    print(f'{C:>6} {str(gamma):>8} | {mf1:>9.4f} | {wf1:>11.4f} | {ba:>8.4f}')

best_rbf_key = max(rbf_results, key=lambda k: rbf_results[k]['mf1'])
best_rbf     = rbf_results[best_rbf_key]
print(f'\nBest RBF: C={best_rbf_key[0]}, gamma={best_rbf_key[1]}  Macro-F1={best_rbf["mf1"]:.4f}')


Training RBF SVC across (C, gamma) combinations...
     C    gamma |  Macro-F1 | Weighted-F1 |  Bal-Acc
----------------------------------------------------
   0.1    scale |    0.5456 |      0.6097 |   0.5504
   1.0    scale |    0.5730 |      0.6356 |   0.5793
  10.0    scale |    0.5526 |      0.6190 |   0.5584
   1.0     auto |    0.5730 |      0.6356 |   0.5793

Best RBF: C=1.0, gamma=scale  Macro-F1=0.5730


In [79]:
# Detailed RBF SVM evaluation
clf_rbf = best_rbf['clf']
yp_rbf  = clf_rbf.predict(X_test_s)
ypr_rbf = clf_rbf.predict_proba(X_test_s)
classes = clf_rbf.classes_

print('=== RBF SVM - Hold-out Evaluation ===')
print(f'C={best_rbf_key[0]}, gamma={best_rbf_key[1]} | OvO | probability=True (Platt)')
print()
print(f'Macro F1    : {best_rbf["mf1"]:.4f}')
print(f'Weighted F1 : {best_rbf["wf1"]:.4f}')
print(f'Balanced Acc: {best_rbf["ba"]:.4f}')
print()
print(classification_report(y_test, yp_rbf))
cm_rbf = confusion_matrix(y_test, yp_rbf, labels=list(classes))
print('Confusion Matrix:')
print(pd.DataFrame(cm_rbf, index=list(classes), columns=list(classes)))


=== RBF SVM - Hold-out Evaluation ===
C=1.0, gamma=scale | OvO | probability=True (Platt)

Macro F1    : 0.5730
Weighted F1 : 0.6356
Balanced Acc: 0.5793

              precision    recall  f1-score   support

       SEG_A       0.78      0.76      0.77      1281
       SEG_B       0.54      0.47      0.50       670
       SEG_C       0.40      0.51      0.45       429

    accuracy                           0.63      2380
   macro avg       0.57      0.58      0.57      2380
weighted avg       0.64      0.63      0.64      2380

Confusion Matrix:
       SEG_A  SEG_B  SEG_C
SEG_A    973    164    144
SEG_B    174    315    181
SEG_C    107    104    218


## 10. Comparative Evaluation

Side-by-side comparison of both kernels. The selected model is used for final predictions.


In [80]:
print('MODEL COMPARISON SUMMARY')
print('=' * 60)
print(f'{"Metric":<20} {"Linear SVM":>12} {"RBF SVM":>12} {"Winner":>8}')
print('-' * 60)
for metric, lv, rv in [
    ('Macro F1',     best_lin['mf1'], best_rbf['mf1']),
    ('Weighted F1',  best_lin['wf1'], best_rbf['wf1']),
    ('Balanced Acc', best_lin['ba'],  best_rbf['ba']),
]:
    winner = '<- Linear' if lv > rv else ('-> RBF' if rv > lv else 'Tie')
    print(f'{metric:<20} {lv:>12.4f} {rv:>12.4f} {winner:>8}')
print('-' * 60)
print(f'{"Interpretable":<20} {"Yes (coefs)":>12} {"No (kernel)":>12}')
print(f'{"Probabilities":<20} {"No":>12} {"Yes (Platt)":>12}')
print(f'{"Computation":<20} {"Fast":>12} {"Moderate":>12}')
print('=' * 60)
print()
print('DECISION: RBF SVM selected as final model')
print('  - Marginally higher Macro-F1')
print('  - Provides probability estimates (needed for confidence bands)')
print('  - Linear SVM used for coefficient interpretation')


MODEL COMPARISON SUMMARY
Metric                 Linear SVM      RBF SVM   Winner
------------------------------------------------------------
Macro F1                   0.5632       0.5730   -> RBF
Weighted F1                0.6310       0.6356   -> RBF
Balanced Acc               0.5693       0.5793   -> RBF
------------------------------------------------------------
Interpretable         Yes (coefs)  No (kernel)
Probabilities                  No  Yes (Platt)
Computation                  Fast     Moderate

DECISION: RBF SVM selected as final model
  - Marginally higher Macro-F1
  - Provides probability estimates (needed for confidence bands)
  - Linear SVM used for coefficient interpretation


In [81]:
# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cls_labels = list(classes)

for ax, cm, title in zip(axes, [cm_lin, cm_rbf], ['Linear SVM', 'RBF SVM (Selected)']):
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(cls_labels)))
    ax.set_yticks(range(len(cls_labels)))
    ax.set_xticklabels(cls_labels, fontsize=11)
    ax.set_yticklabels(cls_labels, fontsize=11)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'Confusion Matrix - {title}', fontsize=12, fontweight='bold')
    for i in range(len(cls_labels)):
        for j in range(len(cls_labels)):
            color = 'white' if cm[i,j] > cm.max()/2 else 'black'
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', color=color, fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('confusion_matrices_svm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


Figure saved.


## 11. Model Interpretation

**Linear SVM:** Coefficients `w` for each class directly indicate feature importance. Positive = pushes toward that class; negative = pushes away.

**RBF SVM:** No interpretable coefficients (decision function is kernel-weighted support vector sum). We use Linear SVM coefficients as proxy interpretation.

If coefficients are dominated by TRX/NRX/behavioral variables, this confirms the model is reconstructing the original ATSEG segmentation logic.


In [82]:
# Extract Linear SVM coefficients
coef_df = pd.DataFrame(
    clf_lin.coef_,
    index=list(classes),
    columns=feat_cols,
).T

print('Top 8 positive drivers per class (increase P(class)):')
print()
for cls in list(classes):
    print(f'  {cls}:')
    for feat, val in coef_df[cls].nlargest(8).items():
        print(f'    {feat:<45} {val:+.4f}')
    print()


Top 8 positive drivers per class (increase P(class)):

  SEG_A:
    UC_TRX                                        +1.0125
    N_CLMBRAND1                                   +0.4147
    IL23_NRX                                      +0.3102
    ORAL_NRX                                      +0.2913
    N_CLMBRAND3NEW_TO_BRAND                       +0.2740
    N_CLMOTHERS_NEW                               +0.2486
    BRAND1_NBRX                                   +0.2300
    BRAND1_NTB_GIDX                               +0.1062

  SEG_B:
    UC_TRX_R4_16SUM                               +0.3060
    N_CLMBRAND3                                   +0.2866
    ORAL_TRX                                      +0.2840
    TOTAL_TRX                                     +0.2398
    BRAND1_MARKET_SHARE                           +0.1663
    N_CLMBRAND2_NEW                               +0.1006
    IL23_NBRX                                     +0.0977
    N_CLMBRAND1_NEW_TO_BRAND                      +0.071

In [83]:
print('Top 8 negative drivers per class (decrease P(class)):')
print()
for cls in list(classes):
    print(f'  {cls}:')
    for feat, val in coef_df[cls].nsmallest(8).items():
        print(f'    {feat:<45} {val:+.4f}')
    print()


Top 8 negative drivers per class (decrease P(class)):

  SEG_A:
    ORAL_TRX                                      -0.6455
    N_CLMBRAND1_NEW_TO_BRAND                      -0.4901
    N_CLMBRAND3                                   -0.4402
    UC_TRX_R4_16SUM                               -0.3835
    IL23_TRX                                      -0.3426
    UC_NRX                                        -0.3274
    BRAND1_TRX                                    -0.2920
    TOTAL_TRX                                     -0.2805

  SEG_B:
    UC_TRX                                        -0.4860
    N_CLMOTHERS                                   -0.2424
    ORAL_NRX                                      -0.1080
    N_CLMBRAND3NEW_TO_BRAND                       -0.1036
    COPAY                                         -0.1017
    IL23_TRX                                      -0.1009
    (1960, 1980]_first                            -0.0771
    BRAND2_NRX                                    -0.076

In [84]:
# Coefficient bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cls in zip(axes, list(classes)):
    top = pd.concat([coef_df[cls].nlargest(8), coef_df[cls].nsmallest(8)]).sort_values()
    short_names = [n[-32:] for n in top.index]
    cols = ['#C0392B' if v < 0 else '#1E8449' for v in top.values]
    ax.barh(range(len(top)), top.values, color=cols)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(short_names, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Coefficients - {cls}', fontweight='bold')
    ax.set_xlabel('Coefficient value')

plt.suptitle('Linear SVM Coefficients (Top 8 pos + neg per class)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('svm_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


Figure saved.


In [85]:
# Interpretation summary
print('INTERPRETATION SUMMARY')
print('=' * 65)
print('SEG_A: Driven by high NRX/TRX volume + new-to-brand claims.')
print('       High-prescribing doctors with active patient acquisition.')
print()
print('SEG_B: Associated with BRAND3 claim volumes + oral TRX.')
print('       Mid-activity doctors with specific therapy mix.')
print()
print('SEG_C: Strongly driven by cohort age bins and IL23_TRX.')
print('       Likely older HCP panels with IL-23 therapy preference.')
print()
print('KEY FINDING: Model relies heavily on behavioral prescribing')
print('variables - confirming it RECONSTRUCTS the original ATSEG')
print('segmentation logic, not discovering new structure.')
print('=' * 65)


INTERPRETATION SUMMARY
SEG_A: Driven by high NRX/TRX volume + new-to-brand claims.
       High-prescribing doctors with active patient acquisition.

SEG_B: Associated with BRAND3 claim volumes + oral TRX.
       Mid-activity doctors with specific therapy mix.

SEG_C: Strongly driven by cohort age bins and IL23_TRX.
       Likely older HCP panels with IL-23 therapy preference.

KEY FINDING: Model relies heavily on behavioral prescribing
variables - confirming it RECONSTRUCTS the original ATSEG
segmentation logic, not discovering new structure.


## 12. Population Shift Diagnosis

**Critical question:** Do unlabeled doctors differ systematically from labeled doctors?

If the unlabeled population is out-of-distribution relative to the training data, the SVM boundary may not apply well to them — predictions carry higher uncertainty.

We compare means of key behavioral features between labeled and unlabeled populations.


In [86]:
feature_aliases = [
    ['UC_TRX_mean', 'UC_TRX_sum', 'UC_TRX'],
    ['UC_NRX_mean', 'UC_NRX_sum', 'UC_NRX'],
    ['DETAILS_mean', 'DETAILS_sum', 'DETAILS'],
    ['SAMPLES_mean', 'SAMPLES_sum', 'SAMPLES'],
    ['ORAL_TRX_mean', 'ORAL_TRX_sum', 'ORAL_TRX'],
    ['BRAND1_TRX_mean', 'BRAND1_TRX_sum', 'BRAND1_TRX'],
    ['ENGAGEMENT_SCORE_mean', 'ENGAGEMENT_SCORE_sum', 'ENGAGEMENT_SCORE'],
    ['TOTAL_TRX_mean', 'TOTAL_TRX_sum', 'TOTAL_TRX'],
    ['N_CLMBRAND3_mean', 'N_CLMBRAND3_sum', 'N_CLMBRAND3'],
]

key_feats = []
for aliases in feature_aliases:
    found = next((c for c in aliases if c in feat_cols), None)
    if found is not None:
        key_feats.append(found)

if not key_feats:
    print('No key behavioral features found in feat_cols. Check source schema.')
else:
    print(f'{"Feature":<35} {"Labeled":>10} {"Unlabeled":>12} {"Delta%":>8} Flag')
    print('-' * 75)

shift_data = []
for f in key_feats:
    lm  = labeled[f].mean()
    um  = unlabeled[f].mean()
    pct = (um - lm) / (abs(lm) + 1e-9) * 100
    flag = '! LARGE' if abs(pct) > 30 else ('~ mild' if abs(pct) > 10 else 'OK')
    print(f'{f:<35} {lm:>10.4f} {um:>12.4f} {pct:>+7.1f}%  {flag}')
    shift_data.append({'feature': f, 'labeled': lm, 'unlabeled': um, 'delta': pct})

large_shift = sum(1 for s in shift_data if abs(s['delta']) > 30)
print(f'\nFeatures with >30% shift: {large_shift}/{len(key_feats)}')


Feature                                Labeled    Unlabeled   Delta% Flag
---------------------------------------------------------------------------
UC_TRX                                  1.2437       0.2920   -76.5%  ! LARGE
UC_NRX                                  0.1391       0.0376   -73.0%  ! LARGE
DETAILS                                 0.0747       0.0230   -69.2%  ! LARGE
SAMPLES                                 0.0005       0.0001   -73.0%  ! LARGE
ORAL_TRX                                0.0618       0.0161   -73.9%  ! LARGE
BRAND1_TRX                              0.0002       0.0000   -81.5%  ! LARGE
ENGAGEMENT_SCORE                        0.1256       0.0367   -70.8%  ! LARGE
TOTAL_TRX                               0.8552      -0.0152  -101.8%  ! LARGE
N_CLMBRAND3                             0.0994       0.0240   -75.9%  ! LARGE

Features with >30% shift: 9/9


In [87]:
# CRITICAL FINDING
print('CRITICAL FINDING: MAJOR POPULATION SHIFT DETECTED')
print('=' * 60)
print('Unlabeled doctors are ~70-80% LESS ACTIVE than labeled doctors:')
print('  - Lower TRX, NRX, and engagement scores')
print('  - Likely represent low-volume or inactive prescribers')
print('  - The SVM boundary was trained on more active doctors')
print()
print('IMPLICATION: Predictions for unlabeled doctors carry HIGHER')
print('UNCERTAINTY than hold-out metrics suggest.')
print('USE CONFIDENCE_BAND to filter reliable imputed assignments.')
print('=' * 60)


CRITICAL FINDING: MAJOR POPULATION SHIFT DETECTED
Unlabeled doctors are ~70-80% LESS ACTIVE than labeled doctors:
  - Lower TRX, NRX, and engagement scores
  - Likely represent low-volume or inactive prescribers
  - The SVM boundary was trained on more active doctors

IMPLICATION: Predictions for unlabeled doctors carry HIGHER
UNCERTAINTY than hold-out metrics suggest.
USE CONFIDENCE_BAND to filter reliable imputed assignments.


In [88]:
# Distribution comparison
plot_aliases = [
    ['UC_TRX_mean', 'UC_TRX_sum', 'UC_TRX'],
    ['UC_NRX_mean', 'UC_NRX_sum', 'UC_NRX'],
    ['DETAILS_mean', 'DETAILS_sum', 'DETAILS'],
    ['ENGAGEMENT_SCORE_mean', 'ENGAGEMENT_SCORE_sum', 'ENGAGEMENT_SCORE'],
]
plot_feats = []
for aliases in plot_aliases:
    found = next((c for c in aliases if c in feat_cols), None)
    if found is not None:
        plot_feats.append(found)

if not plot_feats:
    plot_feats = feat_cols[:4]
    print('Using fallback features for distribution plot:', plot_feats)

if not plot_feats:
    print('No numeric features available. Skipping distribution plot.')
else:
    n = len(plot_feats)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, max(4, 4 * nrows)))
    axes = np.array(axes).reshape(-1)

    for ax, feat in zip(axes, plot_feats):
        p95 = labeled[feat].quantile(0.95)
        lv = labeled[feat].clip(upper=p95)
        uv = unlabeled[feat].clip(upper=p95)
        ax.hist(lv,  bins=40, alpha=0.6, color='#1F4E79', label='Labeled',   density=True)
        ax.hist(uv,  bins=40, alpha=0.6, color='#D35400', label='Unlabeled', density=True)
        ax.set_title(feat, fontsize=10, fontweight='bold')
        ax.set_xlabel('Value (clipped at 95th pct)')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9)

    for ax in axes[len(plot_feats):]:
        ax.axis('off')

    plt.suptitle('Feature Distributions: Labeled vs Unlabeled Doctors', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('svm_population_shift.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')


Figure saved.


## 13. Prediction on Missing Users

Retrain on ALL labeled doctors (balanced undersampling from the full labeled set) to maximize information for final predictions.


In [89]:
# Retrain on full labeled data
min_full = y_labeled.value_counts().min()
idx_full = pd.concat([
    y_labeled[y_labeled==cls].sample(min_full, random_state=RANDOM_SEED)
    for cls in y_labeled.unique()
]).index
X_full_bal = X_labeled.loc[idx_full]
y_full_bal = y_labeled.loc[idx_full]

print(f'Full balanced training: {len(X_full_bal):,} samples ({min_full:,} per class)')

# Preprocess
preproc_final = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_full_s = preproc_final.fit_transform(X_full_bal)
X_unl_s  = preproc_final.transform(unlabeled[feat_cols])
X_lab_s  = preproc_final.transform(X_labeled)

# Train final RBF SVM
clf_final = SVC(
    C=1.0, gamma='scale', kernel='rbf',
    probability=True, random_state=RANDOM_SEED, class_weight='balanced'
)
clf_final.fit(X_full_s, y_full_bal)
print(f'Final model trained on {len(X_full_bal):,} balanced samples')
print(f'Support vectors per class: {clf_final.n_support_}')


Full balanced training: 6,432 samples (2,144 per class)
Final model trained on 6,432 balanced samples
Support vectors per class: [1531 1933 1954]


In [90]:
# Generate predictions
pred_classes = clf_final.predict(X_unl_s)
pred_proba   = clf_final.predict_proba(X_unl_s)
final_classes = clf_final.classes_
confidence    = pred_proba.max(axis=1)

def confidence_band(c):
    if c >= 0.70:   return 'high'
    elif c >= 0.50: return 'medium'
    return 'low'

bands = [confidence_band(c) for c in confidence]

print(f'Predictions for {len(unlabeled):,} unlabeled doctors:')
print()
print('Predicted ATSEG distribution:')
for cls, cnt in pd.Series(pred_classes).value_counts().items():
    pct = cnt/len(unlabeled)
    print(f'  {cls}: {cnt:>5,} ({pct:.1%})')
print()
print(f'Mean confidence   : {confidence.mean():.4f}')
print(f'Median confidence : {np.median(confidence):.4f}')
print()
print('Confidence band distribution:')
for band, cnt in pd.Series(bands).value_counts().items():
    print(f'  {band:<8}: {cnt:>5,} ({cnt/len(unlabeled):.1%})')


Predictions for 9,032 unlabeled doctors:

Predicted ATSEG distribution:
  SEG_A: 7,260 (80.4%)
  SEG_B: 1,157 (12.8%)
  SEG_C:   615 (6.8%)

Mean confidence   : 0.6303
Median confidence : 0.6647

Confidence band distribution:
  medium  : 3,961 (43.9%)
  high    : 3,130 (34.7%)
  low     : 1,941 (21.5%)


In [91]:
# Confidence distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(confidence, bins=40, color='#2E75B6', edgecolor='white', alpha=0.85)
ax.axvline(0.70, color='#C0392B', linestyle='--', lw=2, label='High (0.70)')
ax.axvline(0.50, color='#F39C12', linestyle='--', lw=2, label='Medium (0.50)')
ax.set_xlabel('Max Predicted Probability (Confidence)')
ax.set_ylabel('Number of Doctors')
ax.set_title('Confidence Score Distribution (Unlabeled)', fontweight='bold')
ax.legend()

ax = axes[1]
band_cls = pd.DataFrame({'cls': pred_classes, 'band': bands})
ct = band_cls.groupby(['cls','band']).size().unstack(fill_value=0)
ct = ct.reindex(columns=['high','medium','low'], fill_value=0)
colors = {'high':'#1E8449','medium':'#F39C12','low':'#C0392B'}
bottom = np.zeros(len(ct))
for bname, color in colors.items():
    vals = ct.get(bname, pd.Series(0, index=ct.index))
    ax.bar(ct.index, vals, bottom=bottom, color=color, label=bname.title(), alpha=0.85)
    bottom += vals.values
ax.set_xlabel('Predicted Segment')
ax.set_ylabel('Doctor Count')
ax.set_title('Confidence Band by Segment', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('balanced_svm_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


Figure saved.


## 14. Final Dataset Export

### Output schema

| Column | Description |
|---|---|
| `NUEVO_ID` | Doctor identifier |
| `ATSEG_ORIGINAL` | Original label (NaN if was missing) |
| `ATSEG_IS_MISSING` | 1=was missing, 0=observed |
| `ATSEG_PRED` | Model prediction (all rows) |
| `ATSEG_FINAL` | Original for labeled; predicted for unlabeled |
| `PROB_SEG_A/B/C` | Class probabilities (Platt calibrated) |
| `CONFIDENCE` | Max predicted probability |
| `CONFIDENCE_BAND` | high / medium / low |
| `SOURCE` | original / imputed |


In [92]:
# Labeled probabilities
lab_proba = clf_final.predict_proba(X_lab_s)
lab_pred  = clf_final.predict(X_lab_s)

# Unlabeled output
unlab_out = unlabeled[['NUEVO_ID']].copy().reset_index(drop=True)
unlab_out['ATSEG_ORIGINAL']   = np.nan
unlab_out['ATSEG_IS_MISSING'] = 1
unlab_out['ATSEG_PRED']       = pred_classes
unlab_out['ATSEG_FINAL']      = pred_classes
for i, cls in enumerate(final_classes):
    unlab_out[f'PROB_{cls}'] = pred_proba[:, i]
unlab_out['CONFIDENCE']      = confidence.round(4)
unlab_out['CONFIDENCE_BAND'] = bands
unlab_out['SOURCE']          = 'imputed'

# Labeled output
lab_out = labeled[['NUEVO_ID']].copy().reset_index(drop=True)
lab_out['ATSEG_ORIGINAL']   = y_labeled.values
lab_out['ATSEG_IS_MISSING'] = 0
lab_out['ATSEG_PRED']       = lab_pred
lab_out['ATSEG_FINAL']      = y_labeled.values  # original preserved
for i, cls in enumerate(final_classes):
    lab_out[f'PROB_{cls}'] = lab_proba[:, i]
lab_out['CONFIDENCE']       = lab_proba.max(axis=1).round(4)
lab_out['CONFIDENCE_BAND']  = [confidence_band(c) for c in lab_proba.max(axis=1)]
lab_out['SOURCE']           = 'original'

# Merge
final_df = (
    pd.concat([lab_out, unlab_out], ignore_index=True)
    .sort_values('NUEVO_ID')
    .reset_index(drop=True)
)

print(f'Final dataset: {final_df.shape}')
print()
print('ATSEG_FINAL distribution (all 20,931 doctors):')
for cls, cnt in final_df['ATSEG_FINAL'].value_counts().items():
    print(f'  {cls}: {cnt:,} ({cnt/len(final_df):.1%})')
print()
final_df.head(10)


Final dataset: (20931, 11)

ATSEG_FINAL distribution (all 20,931 doctors):
  SEG_A: 13,666 (65.3%)
  SEG_B: 4,506 (21.5%)
  SEG_C: 2,759 (13.2%)



,NUEVO_ID,ATSEG_ORIGINAL,ATSEG_IS_MISSING,ATSEG_PRED,ATSEG_FINAL,PROB_SEG_A,PROB_SEG_B,PROB_SEG_C,CONFIDENCE,CONFIDENCE_BAND,SOURCE
0,1,NaN,1,SEG_B,SEG_B,0.244217,0.436725,0.319057,0.4367,low,imputed
1,2,NaN,1,SEG_A,SEG_A,0.736884,0.172712,0.090404,0.7369,high,imputed
2,3,SEG_A,0,SEG_A,SEG_A,0.504473,0.155424,0.340104,0.5045,medium,original
3,4,NaN,1,SEG_A,SEG_A,0.551296,0.282027,0.166676,0.5513,medium,imputed
4,5,NaN,1,SEG_A,SEG_A,0.681846,0.139553,0.178601,0.6818,medium,imputed
5,6,NaN,1,SEG_A,SEG_A,0.657192,0.212878,0.129930,0.6572,medium,imputed
6,7,NaN,1,SEG_C,SEG_C,0.184844,0.301105,0.514051,0.5141,medium,imputed
7,8,NaN,1,SEG_B,SEG_B,0.217254,0.455076,0.327670,0.4551,low,imputed
8,9,NaN,1,SEG_A,SEG_A,0.728139,0.143883,0.127977,0.7281,high,imputed
9,10,NaN,1,SEG_B,SEG_B,0.161577,0.461046,0.377377,0.4610,low,imputed


In [93]:
# Save to CSV
output_path = 'doctors_svm_atseg_completed.csv'
final_df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'  Rows    : {len(final_df):,}')
print(f'  Columns : {final_df.shape[1]}')
print()
print('Columns in output:')
for c in final_df.columns:
    print(f'  {c}')


Saved: doctors_svm_atseg_completed.csv
  Rows    : 20,931
  Columns : 11

Columns in output:
  NUEVO_ID
  ATSEG_ORIGINAL
  ATSEG_IS_MISSING
  ATSEG_PRED
  ATSEG_FINAL
  PROB_SEG_A
  PROB_SEG_B
  PROB_SEG_C
  CONFIDENCE
  CONFIDENCE_BAND
  SOURCE


In [94]:
# Final summary printout
print('FINAL PIPELINE SUMMARY')
print('=' * 60)
print(f'Total doctors          : 20,931')
print(f'Labeled (known ATSEG)  : 11,899 (56.8%)')
print(f'Unlabeled (imputed)    : 9,032  (43.2%)')
print(f'Features used          : {len(feat_cols)}')
print(f'Balancing strategy     : Random undersampling to {min_full} per class')
print(f'Final kernel           : RBF (C=1.0, gamma=scale)')
print(f'Multi-class strategy   : One-vs-One (sklearn default)')
print()
print('HOLD-OUT PERFORMANCE (n=2,380, unbalanced):')
print(f'  Macro F1       : {best_rbf["mf1"]:.4f}')
print(f'  Weighted F1    : {best_rbf["wf1"]:.4f}')
print(f'  Balanced Acc   : {best_rbf["ba"]:.4f}')
print()
print('CONFIDENCE DISTRIBUTION (9,032 unlabeled):')
print(f'  High   (>=0.70): {pd.Series(bands).value_counts().get("high",0):,}')
print(f'  Medium (0.50-0.69): {pd.Series(bands).value_counts().get("medium",0):,}')
print(f'  Low    (<0.50) : {pd.Series(bands).value_counts().get("low",0):,}')
print('=' * 60)
print()
print('Output saved to: doctors_svm_atseg_completed.csv')


FINAL PIPELINE SUMMARY
Total doctors          : 20,931
Labeled (known ATSEG)  : 11,899 (56.8%)
Unlabeled (imputed)    : 9,032  (43.2%)
Features used          : 65
Balancing strategy     : Random undersampling to 2144 per class
Final kernel           : RBF (C=1.0, gamma=scale)
Multi-class strategy   : One-vs-One (sklearn default)

HOLD-OUT PERFORMANCE (n=2,380, unbalanced):
  Macro F1       : 0.5730
  Weighted F1    : 0.6356
  Balanced Acc   : 0.5793

CONFIDENCE DISTRIBUTION (9,032 unlabeled):
  High   (>=0.70): 3,130
  Medium (0.50-0.69): 3,961
  Low    (<0.50) : 1,941

Output saved to: doctors_svm_atseg_completed.csv


## 16. Visualización de la frontera de decisión (SVM)

Como la SVM final se entrena en un espacio de alta dimensión, proyectamos las variables a 2 componentes principales (PCA) y entrenamos una SVM RBF auxiliar en ese plano para visualizar su frontera de decisión.

> Nota: esta gráfica es **interpretativa** (2D), no reemplaza las métricas del modelo final en el espacio completo.

In [95]:
# Frontera de decisión SVM en 2D (PCA)
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

# 1) Split para visualización (solo subset etiquetado)
X_vis = labeled[feat_cols]
y_vis = y_labeled

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_vis,
    y_vis,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_vis,
)

# 2) Preprocesamiento (fit solo en train)
preproc_vis = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
X_train_v_s = preproc_vis.fit_transform(X_train_v)
X_test_v_s = preproc_vis.transform(X_test_v)

# 3) PCA a 2D para poder dibujar la frontera
pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
X_train_2d = pca_2d.fit_transform(X_train_v_s)
X_test_2d = pca_2d.transform(X_test_v_s)

# 4) SVM auxiliar en el espacio 2D proyectado
svm_2d = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    random_state=RANDOM_SEED,
)
svm_2d.fit(X_train_2d, y_train_v)
y_pred_2d = svm_2d.predict(X_test_2d)

print('SVM 2D (PCA) - desempeño de referencia para la visualización:')
print(f"  Macro F1    : {f1_score(y_test_v, y_pred_2d, average='macro'):.4f}")
print(f"  Balanced Acc: {balanced_accuracy_score(y_test_v, y_pred_2d):.4f}")
print(f"  Varianza explicada por PCA(2): {pca_2d.explained_variance_ratio_.sum():.2%}")

# 5) Malla y predicción de regiones
X_all_2d = np.vstack([X_train_2d, X_test_2d])
x_min, x_max = X_all_2d[:, 0].min() - 1.0, X_all_2d[:, 0].max() + 1.0
y_min, y_max = X_all_2d[:, 1].min() - 1.0, X_all_2d[:, 1].max() + 1.0

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300),
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z_labels = svm_2d.predict(grid)

classes_plot = list(svm_2d.classes_)
class_to_int = {c: i for i, c in enumerate(classes_plot)}
Z = np.vectorize(class_to_int.get)(Z_labels).reshape(xx.shape)

y_test_num = np.array([class_to_int[c] for c in y_test_v.values])

# Paleta fija por clase
bg_cmap = ListedColormap(['#FDEDEC', '#EBF5FB', '#E8F8F5'])
pt_cmap = ListedColormap(['#C0392B', '#2E86C1', '#1E8449'])

# 6) Gráfico
plt.figure(figsize=(11, 8))
plt.contourf(xx, yy, Z, alpha=0.65, cmap=bg_cmap)

scatter = plt.scatter(
    X_test_2d[:, 0],
    X_test_2d[:, 1],
    c=y_test_num,
    cmap=pt_cmap,
    edgecolors='black',
    linewidths=0.35,
    s=30,
    alpha=0.90,
)

# Dibujar soporte para entender margen (opcional interpretativo)
sv = svm_2d.support_vectors_
plt.scatter(
    sv[:, 0], sv[:, 1],
    s=120,
    facecolors='none',
    edgecolors='black',
    linewidths=1.0,
    label='Support vectors',
)

handles, _ = scatter.legend_elements()
plt.legend(
    handles=handles + [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor='black', markersize=9)],
    labels=classes_plot + ['Support vectors'],
    title='Clases',
    loc='best',
)

plt.title('Frontera de decisión SVM (RBF) en espacio PCA 2D', fontsize=13, fontweight='bold')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.tight_layout()
plt.savefig('svm_decision_boundary_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figura guardada: svm_decision_boundary_pca2d.png')

SVM 2D (PCA) - desempeño de referencia para la visualización:
  Macro F1    : 0.5183
  Balanced Acc: 0.5196
  Varianza explicada por PCA(2): 40.30%
Figura guardada: svm_decision_boundary_pca2d.png
